IMPORT THE LIBRARIES:-

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import pytz

In [2]:
import nltk
nltk.download('vader_lexicon')


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\gk442\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

LOAD THE DATASET:-

In [3]:
apps_df = pd.read_csv(r"C:\Users\gk442\Downloads\Play Store Data.csv")
reviews_df = pd.read_csv(r"C:\Users\gk442\Downloads\User Reviews.csv")


TASK 1:- 
We need to create a Grouped Bar Chart.

Conditions:

-  Top 10 app categories by number of installs
-  Show Average Rating
-  Show Total Reviews
-  Average Rating ≥ 4.0
-  App Size ≥ 10M
-  Last Updated month = January
-  Graph should only appear between 3 PM IST and 5 PM IST
-  Otherwise display a message instead of graph.

In [4]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
apps_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='str')

STEP 1:- Data Cleaning

In [6]:
## RATING- Convert Rating into numeric.
apps_df["Rating"] = pd.to_numeric(apps_df["Rating"], errors="coerce")

## REVIEWS-
apps_df["Reviews"] = pd.to_numeric(apps_df["Reviews"], errors="coerce")

We need everything in MB- "size"

In [7]:
def convert_size(size):
    if pd.isna(size):
        return np.nan

    if size == "Varies with device":
        return np.nan

    size = str(size)

    if "M" in size:
        return float(size.replace("M",""))

    elif "k" in size:
        return float(size.replace("k",""))/1024

    else:
        return np.nan

In [8]:
apps_df["Size_MB"] = apps_df["Size"].apply(convert_size)

In [9]:
### Installs - remove comma and .
apps_df["Installs"] = (
    apps_df["Installs"]
    .str.replace(",","", regex=False)
    .str.replace("+","", regex=False)
)

apps_df["Installs"] = pd.to_numeric(apps_df["Installs"], errors="coerce")

In [10]:
### Last Updated - Convert into datetime.
apps_df["Last Updated"] = pd.to_datetime(apps_df["Last Updated"], errors="coerce")

In [11]:
### Extract month.
apps_df["Month"] = apps_df["Last Updated"].dt.month

In [12]:
Month = 1

STEP 2:- Apply Filters

In [13]:
filtered_df = apps_df[
    (apps_df["Rating"] >= 4.0) &
    (apps_df["Size_MB"] >= 10) &
    (apps_df["Month"] == 1)
]

Find Top 10 Categories by Installs
Group by category.

In [14]:
top_categories = (
    filtered_df
    .groupby("Category")["Installs"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [15]:
top_category_names = top_categories.index

 STEP 3:- Aggregate Data
- Need Average Rating Total Reviews

In [16]:
result = (
    filtered_df[
        filtered_df["Category"].isin(top_category_names)
    ]
    .groupby("Category")
    .agg(
        Average_Rating=("Rating","mean"),
        Total_Reviews=("Reviews","sum")
    )
)

In [ ]:
## Reset index
result = result.reset_index()

STEP 4:- Time Restriction (IST):-

In [18]:
### Current time in India
india = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(india)

In [19]:
hour = current_time.hour

Condition:-
3PM =15
5PM =17

In [20]:
if 15 <= hour < 17:

    print("Showing Graph")

else:

    print("Graph Hidden")

Graph Hidden


STEP 5:- Creating Grouped Bar Chart

In [21]:
if 15 <= hour < 17:
    x = np.arange(len(result))
    width = 0.4
    plt.figure(figsize=(14,6))
    plt.bar(
        x-width/2,
        result["Average_Rating"],
        width,
        label="Average Rating"
    )
    plt.bar(
        x+width/2,
        result["Total_Reviews"],
        width,
        label="Total Reviews"
    )
    plt.xticks(
        x,
        result["Category"],
        rotation=45
    )
    plt.title("Top 10 Categories by Installs")
    plt.xlabel("Category")
    plt.ylabel("Values")
    plt.legend()
    plt.tight_layout()
    plt.show()

else:

    print("This chart is available only between 3 PM IST and 5 PM IST.")

This chart is available only between 3 PM IST and 5 PM IST.


TASK 2:- 
Create an Interactive Choropleth Map using Plotly.

Conditions
- Interactive Choropleth Map
- Show global installs by Category
- Only Top 5 Categories (based on installs)
- Highlight categories where installs > 1,000,000
- Category name must not start with A, C, G, or S
- Graph visible only between 6 PM and 8 PM IST
- Otherwise hide the graph.

STEP 1:- Remove Categories Starting with A, C, G, S

In [22]:
apps_df["Installs"].dtype
# Remove categories starting with A, C, G, or S:-

task2_df = apps_df[
    ~apps_df["Category"].str.startswith(("A", "C", "G", "S"), na=False)
].copy()

In [23]:
sorted(task2_df["Category"].unique())

['1.9',
 'BEAUTY',
 'BOOKS_AND_REFERENCE',
 'BUSINESS',
 'DATING',
 'EDUCATION',
 'ENTERTAINMENT',
 'EVENTS',
 'FAMILY',
 'FINANCE',
 'FOOD_AND_DRINK',
 'HEALTH_AND_FITNESS',
 'HOUSE_AND_HOME',
 'LIBRARIES_AND_DEMO',
 'LIFESTYLE',
 'MAPS_AND_NAVIGATION',
 'MEDICAL',
 'NEWS_AND_MAGAZINES',
 'PARENTING',
 'PERSONALIZATION',
 'PHOTOGRAPHY',
 'PRODUCTIVITY',
 'TOOLS',
 'TRAVEL_AND_LOCAL',
 'VIDEO_PLAYERS',
 'WEATHER']

In [24]:
task2_df = task2_df[task2_df["Category"] != "1.9"]

STEP 2:- Calculate Total Installs by Category

In [63]:
## Calculate Total Installs by Category
category_installs = (
    task2_df.groupby("Category", as_index=False)["Installs"]
    .sum()
)

STEP 3:- Select the Top 5 Categories

In [64]:
top5_categories = (
    category_installs
    .sort_values(by="Installs", ascending=False)
    .head(5)
)

top5_categories

,Category,Installs
20,PRODUCTIVITY,1.417609e+10
21,TOOLS,1.145277e+10
7,FAMILY,1.025826e+10
19,PHOTOGRAPHY,1.008825e+10
16,NEWS_AND_MAGAZINES,7.496318e+09


STEP 4:- Keep Categories with More Than 1 Million Installs

In [65]:
top5_categories = top5_categories[
    top5_categories["Installs"] > 1000000
]

top5_categories

,Category,Installs
20,PRODUCTIVITY,1.417609e+10
21,TOOLS,1.145277e+10
7,FAMILY,1.025826e+10
19,PHOTOGRAPHY,1.008825e+10
16,NEWS_AND_MAGAZINES,7.496318e+09


STEP 5:- Create a list of countries

In [28]:
### Step 1: Create a list of countries
countries = [
    "India",
    "United States",
    "Brazil",
    "Canada",
    "Germany",
    "United Kingdom",
    "Australia",
    "Japan",
    "France",
    "South Africa"
]

STEP 6:- Create a synthetic global dataset 

In [66]:
rows = []

for _, row in top5_categories.iterrows():

    percentages = np.random.dirichlet(np.ones(len(countries)))

    installs = (percentages * row["Installs"]).astype(int)

    for country, install in zip(countries, installs):

        rows.append({
            "Country": country,
            "Category": row["Category"],
            "Installs": install
        })

global_df = pd.DataFrame(rows)

global_df.head()

,Country,Category,Installs
0,India,PRODUCTIVITY,2777660865
1,United States,PRODUCTIVITY,1462376618
2,Brazil,PRODUCTIVITY,2569441593
3,Canada,PRODUCTIVITY,1055735790
4,Germany,PRODUCTIVITY,1336308661


STEP 7:- Highlight installs greater than 1 million

Create a new column to identify rows where installs exceed 1,000,000.

In [30]:
global_df["Highlight"] = np.where(
    global_df["Installs"] > 1000000,
    "Above 1 Million",
    "Below 1 Million"
)

global_df.head()

,Country,Category,Installs,Highlight
0,India,PRODUCTIVITY,840407377,Above 1 Million
1,United States,PRODUCTIVITY,2233221391,Above 1 Million
2,Brazil,PRODUCTIVITY,331428749,Above 1 Million
3,Canada,PRODUCTIVITY,1908101825,Above 1 Million
4,Germany,PRODUCTIVITY,304825199,Above 1 Million


STEP 8:- Add the Time Condition (6 PM–8 PM IST)

In [31]:
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)

hour = current_time.hour

In [32]:
import plotly.express as px

STEP 9:- Create the Choropleth Map

In [33]:
if 18 <= hour < 20:

    fig = px.choropleth(
        global_df,
        locations="Country",
        locationmode="country names",
        color="Installs",
        animation_frame="Category",   # Interactive category filter
        hover_name="Country",
        hover_data={
            "Category": True,
            "Installs": ":,",
            "Highlight": True
        },
        color_continuous_scale="Turbo",
        title="🌍 Global Installs by Google Play Store Categories"
    )

    fig.update_layout(
        title=dict(
            text="🌍 Global Installs by Google Play Store Categories",
            x=0.5,
            font=dict(size=24)
        ),
        geo=dict(
            showframe=False,
            showcoastlines=True,
            projection_type="natural earth",
            bgcolor="white"
        ),
        paper_bgcolor="white",
        margin=dict(l=20, r=20, t=70, b=20)
    )

    fig.show()

else:
    print("⏰ This graph is available only between 6:00 PM IST and 8:00 PM IST.")


⏰ This graph is available only between 6:00 PM IST and 8:00 PM IST.


In [34]:
fig = px.choropleth(
    global_df,
    locations="Country",
    locationmode="country names",
    color="Installs",
    animation_frame="Category",   # Interactive category filter
    hover_name="Country",
    hover_data={
        "Category": True,
        "Installs": ":,",
        "Highlight": True
    },
    color_continuous_scale="Turbo",
    title="🌍 Global Installs by Google Play Store Categories"
)

fig.update_layout(
    title=dict(
        text="🌍 Global Installs by Google Play Store Categories",
        x=0.5,
        font=dict(size=24)
    ),
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth",
        bgcolor="white"
    ),
    paper_bgcolor="white",
    margin=dict(l=20, r=20, t=70, b=20)
)

fig.show()

C:\Users\gk442\AppData\Local\Temp\ipykernel_25632\2219897089.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


TASK 3:-
We need to create a dual-axis chart comparing:

- Average Installs
- Revenue
- Free vs Paid Apps
- Only Top 3 Categories

STEP 1:- Create a copy

In [35]:
task3_df = apps_df.copy()

In [36]:
task3_df[["Price","Installs"]].dtypes


Price           str
Installs    float64
dtype: object

STEP 2:- DATA CLEANING

In [37]:
task3_df["Size_MB"] = (
    task3_df["Size"]
    .astype(str)
    .str.replace("M","", regex=False)
    .str.replace("k","", regex=False)
)

task3_df["Size_MB"] = pd.to_numeric(task3_df["Size_MB"], errors="coerce")

In [38]:
task3_df["Price"] = (
    task3_df["Price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.strip()
)

In [39]:
task3_df["Price"] = pd.to_numeric(task3_df["Price"], errors="coerce")

In [40]:
task3_df[["Size","Size_MB"]].head(10)

,Size,Size_MB
0,19M,19.0
1,14M,14.0
2,8.7M,8.7
3,25M,25.0
4,2.8M,2.8
5,5.6M,5.6
6,19M,19.0
7,29M,29.0
8,33M,33.0
9,3.1M,3.1


STEP 3:- Revenue Calculation

In [41]:
task3_df["Revenue"] = task3_df["Installs"] * task3_df["Price"]

In [42]:
task3_df[["App","Price","Installs","Revenue"]].head()

,App,Price,Installs,Revenue
0,Photo Editor & Candy Camera & Grid & ScrapBook,0.0,10000.0,0.0
1,Coloring book moana,0.0,500000.0,0.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",0.0,5000000.0,0.0
3,Sketch - Draw & Paint,0.0,50000000.0,0.0
4,Pixel Draw - Number Art Coloring Book,0.0,100000.0,0.0


STEP4:- Android Version Cleaning

In [43]:
task3_df["Android_Num"] = (
    task3_df["Android Ver"]
    .astype(str)
    .str.extract(r'(\d+\.\d+)')[0]
)

task3_df["Android_Num"] = pd.to_numeric(
    task3_df["Android_Num"],
    errors="coerce"
)

In [44]:
task3_df[["Android Ver","Android_Num"]].head(10)

,Android Ver,Android_Num
0,4.0.3 and up,4.0
1,4.0.3 and up,4.0
2,4.0.3 and up,4.0
3,4.2 and up,4.2
4,4.4 and up,4.4
5,2.3 and up,2.3
6,4.0.3 and up,4.0
7,4.2 and up,4.2
8,3.0 and up,3.0
9,4.0.3 and up,4.0


STEP 5:- Apply All Filters

In [45]:
task3_df = task3_df[
    (task3_df["Installs"] >= 10000) &
    (task3_df["Revenue"] >= 10000) &
    (task3_df["Android_Num"] > 4.0) &
    (task3_df["Size_MB"] > 15) &
    (task3_df["Content Rating"] == "Everyone") &
    (task3_df["App"].str.len() <= 30)
]

task3_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Month,Revenue,Android_Num
853,Toca Life: City,EDUCATION,4.7,31085.0,24M,500000.0,Paid,3.99,Everyone,Education;Pretend Play,2018-07-06,1.5-play,4.4 and up,24.0,7.0,1995000.0,4.4
854,Toca Life: Hospital,EDUCATION,4.7,3528.0,24M,100000.0,Paid,3.99,Everyone,Education;Pretend Play,2018-06-12,1.1.1-play,4.4 and up,24.0,6.0,399000.0,4.4
1335,Meditation Studio,HEALTH_AND_FITNESS,4.6,1026.0,29M,10000.0,Paid,3.99,Everyone,Health & Fitness,2018-05-15,1.0.6,4.3 and up,29.0,5.0,39900.0,4.3
1831,The Game of Life,GAME,4.4,18621.0,63M,100000.0,Paid,2.99,Everyone,Board,2018-07-04,2.1.2,4.4 and up,63.0,7.0,299000.0,4.4
1833,The Room: Old Sins,GAME,4.9,21119.0,48M,100000.0,Paid,4.99,Everyone,Puzzle,2018-04-18,1.0.1,4.4 and up,48.0,4.0,499000.0,4.4


STEP 6:- Find Top 3 Categories

In [46]:
top3_categories = (
    task3_df.groupby("Category", as_index=False)
    .agg({
        "Installs": "sum"
    })
    .sort_values("Installs", ascending=False)
    .head(3)
)

top3_categories

,Category,Installs
8,PHOTOGRAPHY,3000000.0
1,FAMILY,2670000.0
7,PERSONALIZATION,2010000.0


STEP 7:- Keep Only Those Categories

In [47]:
task3_df = task3_df[
    task3_df["Category"].isin(top3_categories["Category"])
]

task3_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Month,Revenue,Android_Num
2151,Toca Life: City,FAMILY,4.7,31100.0,24M,500000.0,Paid,3.99,Everyone,Education;Pretend Play,2018-07-06,1.5-play,4.4 and up,24.0,7.0,1995000.0,4.4
2883,Facetune - For Free,PHOTOGRAPHY,4.4,49553.0,48M,1000000.0,Paid,5.99,Everyone,Photography,2018-07-25,1.3.1,4.1 and up,48.0,7.0,5990000.0,4.1
2912,Facetune - For Free,PHOTOGRAPHY,4.4,49553.0,48M,1000000.0,Paid,5.99,Everyone,Photography,2018-07-25,1.3.1,4.1 and up,48.0,7.0,5990000.0,4.1
2950,Facetune - For Free,PHOTOGRAPHY,4.4,49553.0,48M,1000000.0,Paid,5.99,Everyone,Photography,2018-07-25,1.3.1,4.1 and up,48.0,7.0,5990000.0,4.1
3405,HD Widgets,PERSONALIZATION,4.3,58617.0,26M,1000000.0,Paid,0.99,Everyone,Personalization,2016-12-07,4.3.2,4.4 and up,26.0,12.0,990000.0,4.4


In [48]:
task3_df.shape

(13, 17)

STEP 8:- Prepare Data for the Dual-Axis Chart

We need:

Average Installs
Average Revenue
Compare Free vs Paid Apps
Only for Top 3 Categories

In [49]:
summary = (
    task3_df
    .groupby(["Category","Type"])
    .agg(
        Avg_Installs=("Installs","mean"),
        Revenue=("Revenue","mean")
    )
    .reset_index()
)

summary

,Category,Type,Avg_Installs,Revenue
0,FAMILY,Paid,381428.571429,7.969000e+05
1,PERSONALIZATION,Paid,670000.000000,6.666333e+05
2,PHOTOGRAPHY,Paid,1000000.000000,5.990000e+06


Create the Dual-Axis Plotly Chart

In [50]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [51]:
from datetime import datetime
import pytz

ist = pytz.timezone("Asia/Kolkata")
hour = datetime.now(ist).hour

if 13 <= hour < 14:

    print("Chart Visible")

else:

    print("Chart Hidden")

Chart Hidden


In [52]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

ist = pytz.timezone("Asia/Kolkata")
hour = datetime.now(ist).hour

if 13 <= hour < 14:

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Average Installs (Bar Chart)
    for app_type in summary["Type"].unique():

        temp = summary[summary["Type"] == app_type]

        fig.add_trace(
            go.Bar(
                x=temp["Category"],
                y=temp["Avg_Installs"],
                name=f"{app_type} Avg Installs"
            ),
            secondary_y=False
        )

    # Revenue (Line Chart)
    for app_type in summary["Type"].unique():

        temp = summary[summary["Type"] == app_type]

        fig.add_trace(
            go.Scatter(
                x=temp["Category"],
                y=temp["Revenue"],
                mode="lines+markers",
                name=f"{app_type} Revenue"
            ),
            secondary_y=True
        )

    fig.update_layout(
        title="Average Installs vs Revenue (Free vs Paid Apps)",
        xaxis_title="Top 3 Categories",
        template="plotly_white",
        width=1000,
        height=600
    )

    fig.update_yaxes(title_text="Average Installs", secondary_y=False)
    fig.update_yaxes(title_text="Revenue ($)", secondary_y=True)

    fig.show()

else:
    print("⏰ This graph is available only between 1 PM IST and 2 PM IST.")

⏰ This graph is available only between 1 PM IST and 2 PM IST.


now this without timeline

In [53]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Average Installs (Bar Chart)
for app_type in summary["Type"].unique():

    temp = summary[summary["Type"] == app_type]

    fig.add_trace(
        go.Bar(
            x=temp["Category"],
            y=temp["Avg_Installs"],
            name=f"{app_type} Avg Installs"
        ),
        secondary_y=False
    )

# Revenue (Line Chart)
for app_type in summary["Type"].unique():

    temp = summary[summary["Type"] == app_type]

    fig.add_trace(
        go.Scatter(
            x=temp["Category"],
            y=temp["Revenue"],
            mode="lines+markers",
            name=f"{app_type} Revenue"
        ),
        secondary_y=True
    )

fig.update_layout(
    title="Average Installs vs Revenue (Free vs Paid Apps)",
    xaxis_title="Top 3 Categories",
    template="plotly_white",
    width=1000,
    height=600
)

fig.update_yaxes(title_text="Average Installs", secondary_y=False)
fig.update_yaxes(title_text="Revenue ($)", secondary_y=True)

fig.show()